In [ ]:
# BETA TUNING

from simulator import run_pytorch_monte_carlo
from util import create_final_matrix
from helpers import (
    calculate_market_rmse,
    safe_scrape_elo,
    country_to_elo,
)
import numpy as np
from constants import TEAM_ID_MAP, ALPHA
import pandas as pd

elo_df = safe_scrape_elo()


# format: [Home, Away, Home_Elo, Away_Elo, Bookie_Odds(H, D, A)]
validation_set = [
    (
        "Switzerland",
        "Colombia",
        country_to_elo(elo_df, "Switzerland"),
        country_to_elo(elo_df, "Colombia"),
        [3.90, 3.19, 2.48],
    ),
    (
        "Argentina",
        "Egypt",
        country_to_elo(elo_df, "Argentina"),
        country_to_elo(elo_df, "Egypt"),
        [1.38, 5.07, 13.00],
    ),
    (
        "USA",
        "Belgium",
        country_to_elo(elo_df, "USA"),
        country_to_elo(elo_df, "Belgium"),
        [2.62, 3.50, 2.98],
    ),
    (
        "Portugal",
        "Spain",
        country_to_elo(elo_df, "Portugal"),
        country_to_elo(elo_df, "Spain"),
        [4.12, 3.67, 2.06],
    ),
    (
        "Mexico",
        "England",
        country_to_elo(elo_df, "Mexico"),
        country_to_elo(elo_df, "England"),
        [3.23, 3.37, 2.60],
    ),
    (
        "Brazil",
        "Norway",
        country_to_elo(elo_df, "Brazil"),
        country_to_elo(elo_df, "Norway"),
        [1.85, 3.86, 5.07],
    ),
]

beta_candidates = np.linspace(start=0.0001, stop=0.0005, num=12)

best_beta = None
lowest_error = 999.0

global_q_matrix = pd.read_csv("./global_q_matrix.csv")
global_q_grid = pd.read_csv("./global_q_grid.csv")

print("[*] Beginning Beta Grid Search...")

for beta in beta_candidates:
    total_rmse = 0.0
    for match in validation_set:
        home, away, elo_h, elo_a, odds = match
        test_q_grid = create_final_matrix(
            home,
            TEAM_ID_MAP[home],
            away,
            TEAM_ID_MAP[away],
            elo_h,
            elo_a,
            global_q_matrix,
            ALPHA,
            beta,
        )

        p_h, p_d, p_a = run_pytorch_monte_carlo(test_q_grid, num_simulations=5000)
        model_probs = [p_h, p_d, p_a]
        error = calculate_market_rmse(model_probs, odds)
        total_rmse += error

    avg_rmse = total_rmse / len(validation_set)
    print(f"Beta: {beta:.6f} | Average Market RMSE: {avg_rmse:.4f}")

    if avg_rmse < lowest_error:
        best_beta = beta
        lowest_error = avg_rmse
        print(f"New best beta found: {best_beta}!")
print(f"\n[+] Optimal Beta Found: {best_beta:.6f} (Error: {lowest_error:.4f})")

In [ ]:
# SIMULTANEOUS ALPHA AND BETA TUNING
# this must be run on my GPU since it'll take 38 min on my laptop


from simulator import run_pytorch_monte_carlo
from util import create_final_matrix
from helpers import (
    calculate_market_rmse,
    safe_scrape_elo,
    country_to_elo,
)
import numpy as np
from constants import TEAM_ID_MAP
import pandas as pd
from tqdm import tqdm

elo_df = safe_scrape_elo()


# format: [Home, Away, Home_Elo, Away_Elo, Bookie_Odds(H, D, A)]
validation_set = [
    (
        "Switzerland",
        "Colombia",
        country_to_elo(elo_df, "Switzerland"),
        country_to_elo(elo_df, "Colombia"),
        [3.90, 3.19, 2.48],
    ),
    (
        "Argentina",
        "Egypt",
        country_to_elo(elo_df, "Argentina"),
        country_to_elo(elo_df, "Egypt"),
        [1.38, 5.07, 13.00],
    ),
    (
        "USA",
        "Belgium",
        country_to_elo(elo_df, "USA"),
        country_to_elo(elo_df, "Belgium"),
        [2.62, 3.50, 2.98],
    ),
    (
        "Portugal",
        "Spain",
        country_to_elo(elo_df, "Portugal"),
        country_to_elo(elo_df, "Spain"),
        [4.12, 3.67, 2.06],
    ),
    (
        "Mexico",
        "England",
        country_to_elo(elo_df, "Mexico"),
        country_to_elo(elo_df, "England"),
        [3.23, 3.37, 2.60],
    ),
    (
        "Brazil",
        "Norway",
        country_to_elo(elo_df, "Brazil"),
        country_to_elo(elo_df, "Norway"),
        [1.85, 3.86, 5.07],
    ),
]
alpha_candidates = np.linspace(start=0.01, stop=0.30, num=10)
beta_candidates = np.linspace(start=0.0001, stop=0.0006, num=10)

total_iterations = len(alpha_candidates) * len(beta_candidates)

best_alpha = None
best_beta = None
lowest_error = 999.0

global_q_matrix = pd.read_csv("./global_q_matrix.csv")
global_q_grid = pd.read_csv("./global_q_grid.csv")

print(
    f"[*] Beginning 2D Grid Search ({len(alpha_candidates) * len(beta_candidates)} combos)..."
)
with tqdm(
    total=total_iterations, desc="Optimising Hyperparameters", unit="grid"
) as pbar:
    for alpha in alpha_candidates:
        for beta in beta_candidates:
            total_rmse = 0.0

            for match in validation_set:
                home, away, elo_h, elo_a, odds = match

                # Pass BOTH dynamic hyperparameters
                test_q_grid = create_final_matrix(
                    home,
                    TEAM_ID_MAP[home],
                    away,
                    TEAM_ID_MAP[away],
                    elo_h,
                    elo_a,
                    global_q_matrix,
                    alpha,
                    beta,
                )

                p_h, p_d, p_a = run_pytorch_monte_carlo(
                    test_q_grid, num_simulations=3000
                )

                model_probs = [p_h, p_d, p_a]

                error = calculate_market_rmse(model_probs, odds)
                total_rmse += error

            avg_rmse = total_rmse / len(validation_set)

            if avg_rmse < lowest_error:
                tqdm.write(
                    f"[!] New best found! Alpha: {alpha:.4f} | Beta: {beta:.6f} | RMSE: {avg_rmse:.4f}"
                )
                lowest_error = avg_rmse
                best_alpha = alpha
                best_beta = beta
            pbar.update(1)

print("\n=========================================")
print("          OPTIMAL HYPERPARAMETERS        ")
print("=========================================")
print(f"Alpha: {best_alpha:.4f}")
print(f"Beta:  {best_beta:.6f}")
print(f"Final Market RMSE: {lowest_error:.4f}")
print("=========================================\n")